# Fine-tune Qwen3-VL-8B to box circuit symbols (QLoRA, Kaggle, CGHD)

Teaches the model the one job the schematic reader's `wires.locate()` asks of it: a tight `bbox_2d` around every
component symbol in a hand-drawn circuit, labelled with its kind and the designator/value written beside it.
Values and wiring are still read by the pipeline as before.

**Kaggle setup:** Settings -> Accelerator = GPU (T4 or P100; Unsloth uses one GPU), Internet = On. Add two datasets:
1. **CGHD** (*A Public Ground-Truth Dataset for Handwritten Circuit Diagrams*, Thoma et al. 2021, CC BY-SA 3.0):
   the `archive.zip` with `classes.json` and `drafter_*/` at its top level.
2. **The reader's scripts:** upload `training/schematic-reader-scripts.zip` (a copy of
   `context/schematic-reader/scripts`) as a private dataset. The notebook prepares every image with the reader's own
   code and asks the reader's own question, so training sees exactly what the pipeline sends.

Both are found by searching `/kaggle/input`, whatever their slugs. Output goes to `/kaggle/working`: a LoRA adapter,
a merged 16-bit model, and the prepared images (re-used if the session restarts).

In [ ]:
# Run this first. If Kaggle then asks to restart the kernel, restart and carry on from the next cell
# (installed packages survive a restart; variables do not).
# Versions from Unsloth's own Qwen3-VL (8B) notebook, which trains this model on a T4.
!pip install -q unsloth ollama          # ollama: only so the reader's analyze.py imports
!pip install -q transformers==4.57.6
!pip install -q --no-deps trl==0.22.2

In [ ]:
import glob, os, sys
INPUT = "/kaggle/input"
DATA_ROOT = os.path.dirname(next(iter(glob.glob(f"{INPUT}/**/classes.json", recursive=True)), "/missing/x"))
READER = os.path.dirname(next(iter(glob.glob(f"{INPUT}/**/wires.py", recursive=True)), "/missing/x"))
BASE = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"   # public 4-bit Qwen3-VL-8B (checked on Hugging Face 2026-10-29)
OUT = "/kaggle/working"
MAX_SIDE = 1000          # images are prepared exactly as the reader does, at this --max-side (reader default: 1600)
PRESHRINK = 2000         # phone photos are first shrunk to this long side, only to keep preparation fast
VAL_DRAFTERS = {29, 30, 31}   # whole drafters held out: the 4 photos of a drawing are near-copies of each other
SKIP_DRAFTERS = {-1}     # drafter_-1 has placeholder <size> tags (1000x1000 on 4000 px photos)
PHOTOS_PER_DRAWING = 4   # CGHD has 4 photos (P1-P4) of every drawing; 2 halves training time
MAX_SYMBOLS = 60         # larger sheets make answers longer than the sequence limit; the reader is for small circuits
EPOCHS = 1
SEED = 0
print("dataset:", DATA_ROOT, "| reader scripts:", READER)

## 1. The dataset, as it really is
Checked against the full CGHD release (3,269 annotation files, 33 drafters):
- **Annotations:** Pascal VOC `.xml` per image, in `drafter_D/annotations/`; images in `drafter_D/images/` with mixed
  suffixes (`.jpg .JPG .jpeg .png`). `drafter_0` has 1,037 images but only 149 annotated; only annotated ones are used.
- **Orientation:** boxes are in the **EXIF-upright** frame (phone photos stored sideways have their `<size>` swapped).
  The reader also turns images upright, so `ImageOps.exif_transpose` comes first.
- **Sizes:** `<size>` is not trusted: `drafter_-1` says 1000x1000 on 3968x2976 photos, so it is skipped, and any image
  whose boxes do not fit it is dropped.
- **Classes:** 59 in use; text (91k), junction (79k), terminal and crossover are not symbols. Complex parts
  (transformer, relay, optocoupler) are boxed both whole and as sub-symbols; only the sub-symbols are kept.
- **Texts:** `text` boxes carry their characters (Ω and µ as real characters). They are not tied to symbols, so each
  designator/value text is given to the nearest symbol it fits (see `symbols()`); bare numbers (mostly IC pins) are ignored.
- **No rotation tags** in this release, although the CGHD README mentions them.
- **Splits:** every drawing is photographed 4 times (P1-P4), so a random split leaks near-copies. Whole drafters are held out.

In [ ]:
sys.path.insert(0, READER)
try:
    import ollama
except ImportError:  # analyze.py imports it at the top; training never talks to Ollama, so a stand-in will do
    import types
    sys.modules["ollama"] = types.ModuleType("ollama")
from analyze import prepare                      # the reader's own image preparation (ruled lines, crop, scale)
from wires import SYSTEM, GROUNDING, locate_prompt   # and its exact question

In [ ]:
# CGHD class -> the reader's category word (as wires.locate() lists it: schema category with "_" -> " ").
KIND = {
    "resistor": "resistor", "resistor.adjustable": "resistor", "resistor.photo": "resistor", "varistor": "resistor",
    "capacitor.unpolarized": "capacitor", "capacitor.polarized": "capacitor", "capacitor.adjustable": "capacitor",
    "inductor": "inductor", "inductor.ferrite": "inductor", "inductor.coupled": "inductor",
    "diode": "diode", "diode.zener": "diode", "diode.thyrector": "diode", "diode.light_emitting": "led",
    "thyristor": "thyristor", "triac": "thyristor", "diac": "thyristor",
    "transistor.bjt": "transistor", "transistor.photo": "transistor", "transistor.fet": "mosfet",
    "operational_amplifier": "ic", "operational_amplifier.schmitt_trigger": "ic", "integrated_circuit": "ic",
    "integrated_circuit.ne555": "ic", "integrated_circuit.voltage_regulator": "ic",
    "and": "ic", "or": "ic", "not": "ic", "nand": "ic", "nor": "ic", "xor": "ic",
    "voltage.dc": "voltage source", "voltage.ac": "voltage source", "voltage.battery": "battery",
    "gnd": "ground", "vss": "ground",
    "probe": "meter", "probe.current": "meter", "probe.voltage": "meter",
    "switch": "switch", "socket": "connector", "fuse": "fuse", "lamp": "lamp", "crystal": "crystal",
    # the reader files these under "other" and lists them by name
    "speaker": "speaker", "motor": "motor", "microphone": "microphone", "antenna": "antenna"}
# Not component symbols, or the outline of a complex part whose sub-symbols (inductors, switch, LED...)
# are annotated separately: one box per symbol, as the reader asks.
SKIP = {"text", "junction", "crossover", "terminal", "unknown", "block", "explanatory",
        "mechanical", "magnetic", "optical", "transformer", "relay", "optocoupler"}

In [ ]:
import re, json, collections, xml.etree.ElementTree as ET
from PIL import Image, ImageOps

DESIGNATOR = re.compile(r"[A-Za-z]{1,3}\d{1,3}")
VALUE = re.compile(r"\d+(?:[.,]\d+)?\s*(?:[pnuµμmkKM]|Meg)?\s*(?:Ω|[Oo]hm|F|H|V|A|Hz|W)?")

def is_value(t):
    """A component value: a number with a unit or prefix ('10K', '100nF', '9V'). Bare numbers are mostly pin numbers."""
    return bool(VALUE.fullmatch(t)) and not t.replace(".", "").replace(",", "").isdigit()

# A text is only given to a symbol it can belong to: designator letters per kind, and values only for kinds
# that carry one (so "100µF" beside a ground symbol stays with its capacitor or is dropped).
PREFIX = {"resistor": ("R", "VR", "P"), "capacitor": ("C",), "inductor": ("L",), "diode": ("D", "Z"),
          "led": ("D", "LED", "L", "H"), "transistor": ("Q", "T"), "mosfet": ("Q", "T", "M"), "ic": ("U", "IC", "N", "A"),
          "switch": ("S", "SW", "JP", "K"), "battery": ("B", "BT", "V", "G"), "voltage source": ("V", "U", "G", "E"),
          "fuse": ("F",), "lamp": ("L", "H", "E"), "crystal": ("X", "Y", "Q"), "connector": ("J", "X", "P", "K"),
          "thyristor": ("D", "T", "Q", "V"), "meter": ("M", "A", "V", "P"), "speaker": ("LS", "SP", "S"),
          "microphone": ("MIC", "M"), "motor": ("M",), "ground": ()}
VALUED = {"resistor", "capacitor", "inductor", "battery", "voltage source", "fuse", "lamp", "crystal", "led"}

def fits(kind, slot, t):
    if slot == 3:
        return kind in VALUED
    letters = re.match(r"[A-Za-z]+", t).group(0).upper()
    return any(letters.startswith(p) for p in PREFIX.get(kind, ("",)))

def gap(a, b):
    return max(0, a[0] - b[2], b[0] - a[2], a[1] - b[3], b[1] - a[3])

def read(xml_path):
    """[(cghd class, box, text)] of one annotation file."""
    out = []
    for o in ET.parse(xml_path).getroot().iter("object"):
        b = o.find("bndbox")
        out.append((o.findtext("name").strip(), tuple(float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")),
                    (o.findtext("text") or "").strip()))
    return out

def symbols(objs):
    """[(kind, label, box)]: every component symbol, labelled as the reader asks: kind, then the designator
    and value written beside it. CGHD does not tie texts to symbols, so each text goes to the nearest symbol
    within 0.6 of that symbol's size."""
    syms = [[KIND[n], b, None, None] for n, b, _ in objs if n in KIND]
    for n, b, t in objs:
        if n != "text" or not syms:
            continue
        slot = 2 if DESIGNATOR.fullmatch(t) else 3 if is_value(t) else None
        if slot is None:
            continue
        s = min(syms, key=lambda s: gap(s[1], b))   # the nearest symbol only: never pass a text further along
        d = gap(s[1], b)
        if fits(s[0], slot, t) and d <= 0.6 * max(s[1][2] - s[1][0], s[1][3] - s[1][1]) \
                and (s[slot] is None or d < s[slot][0]):
            s[slot] = (d, t)
    return [(k, " ".join([k] + [x[1] for x in (des, val) if x]), b) for k, b, des, val in syms]

def index_images(root):
    """drafter number -> {lower-case file stem: image path}. Suffixes vary (.jpg .JPG .jpeg .png)."""
    idx = collections.defaultdict(dict)
    for p in glob.glob(os.path.join(root, "drafter_*", "images", "*")):
        d = int(re.search(r"drafter_(-?\d+)", p).group(1))
        idx[d][os.path.splitext(os.path.basename(p))[0].lower()] = p
    return idx

In [ ]:
def prepare_one(job):
    """Prepare one image as the reader does and move its boxes along. Returns None for an unusable sample."""
    xml_path, img_path, cache = job
    if os.path.exists(cache + ".json"):
        return json.load(open(cache + ".json"))
    objs = read(xml_path)
    unknown = sorted({n for n, *_ in objs if n not in KIND and n not in SKIP})
    im = ImageOps.exif_transpose(Image.open(img_path))   # CGHD boxes are in the EXIF-upright frame
    W, H = im.size
    syms = symbols(objs)
    # Annotation does not fit the image (wrong size/orientation): skip rather than teach shifted boxes.
    if not syms or any(b[2] > 1.01 * W or b[3] > 1.01 * H for _, _, b in syms):
        return {"skip": "boxes do not fit the image" if syms else "no symbols", "unknown": unknown}
    if len(syms) > MAX_SYMBOLS:
        return {"skip": "too many symbols", "unknown": unknown}
    s0 = min(1.0, PRESHRINK / max(W, H))
    if s0 < 1:
        im = im.resize((round(W * s0), round(H * s0)), Image.LANCZOS)
    im, (x0, y0, s) = prepare(im, MAX_SIDE)
    w, h = im.size
    boxes = []
    for k, label, b in syms:
        x1, y1, x2, y2 = ((b[0] * s0 - x0) * s, (b[1] * s0 - y0) * s, (b[2] * s0 - x0) * s, (b[3] * s0 - y0) * s)
        cx1, cy1, cx2, cy2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
        if (cx2 - cx1) * (cy2 - cy1) < 0.5 * (x2 - x1) * (y2 - y1):
            return {"skip": "the reader's crop cut a symbol off", "unknown": unknown}
        boxes.append({"kind": k, "label": label, "bbox_2d": [round(cx1 / w * 1000), round(cy1 / h * 1000),
                                                             round(cx2 / w * 1000), round(cy2 / h * 1000)]})
    im.save(cache + ".png")
    rec = {"image": cache + ".png", "boxes": boxes, "unknown": unknown}
    json.dump(rec, open(cache + ".json", "w"))
    return rec

def collect(root, cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    idx = index_images(root)
    jobs, meta = [], []
    for x in sorted(glob.glob(os.path.join(root, "drafter_*", "annotations", "*.xml"))):
        d = int(re.search(r"drafter_(-?\d+)", x).group(1))
        stem = os.path.splitext(os.path.basename(x))[0]
        m = re.search(r"_P(\d+)$", stem)
        if d in SKIP_DRAFTERS or (m and int(m.group(1)) > PHOTOS_PER_DRAWING) or stem.lower() not in idx[d]:
            continue
        jobs.append((x, idx[d][stem.lower()], os.path.join(cache_dir, f"{d}_{stem}")))
        meta.append(d)
    return jobs, meta

## 2. Prepare every image as the reader does
Runs on all CPU cores; results are cached in `/kaggle/working/prepared`.

In [ ]:
from multiprocessing import Pool
jobs, drafters = collect(DATA_ROOT, f"{OUT}/prepared")
with Pool(os.cpu_count()) as pool:
    recs = pool.map(prepare_one, jobs, chunksize=4)
print(len(jobs), "annotated images:", collections.Counter(r.get("skip", "used") for r in recs))
unknown = {u for r in recs for u in r["unknown"]}
if unknown:
    print("CGHD classes in neither KIND nor SKIP (ignored, add them to KIND if they are symbols):", unknown)
data = [(d, r) for d, r in zip(drafters, recs) if "skip" not in r]
train_recs = [r for d, r in data if d not in VAL_DRAFTERS]
val_recs = [r for d, r in data if d in VAL_DRAFTERS]
print(len(train_recs), "train /", len(val_recs), "held out (drafters", sorted(VAL_DRAFTERS), ")")
print(collections.Counter(b["kind"] for r in train_recs for b in r["boxes"]).most_common())

## 3. Chat examples in the pipeline's own format
System prompt, question (`locate_prompt()` + `GROUNDING`) and answer format are imported from `wires.py`, the same
text `wires.locate()` sends. Coordinates are normalised to 0-1000 of the prepared image.

In [ ]:
def example(rec):
    boxes = sorted(rec["boxes"], key=lambda b: (b["bbox_2d"][1] // 100, b["bbox_2d"][0]))  # reading order
    kinds = sorted({b["kind"] for b in boxes})
    answer = "[\n" + ",\n".join('  {"bbox_2d": %s, "label": %s}' % (json.dumps(b["bbox_2d"]),
                                                                      json.dumps(b["label"], ensure_ascii=False))
                                   for b in boxes) + "\n]"
    return {"messages": [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "image", "image": Image.open(rec["image"]).convert("RGB")},
                                     {"type": "text", "text": f"{locate_prompt(kinds)}\n{GROUNDING}"}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]}]}

import random
random.seed(SEED)
random.shuffle(train_recs)
train = [example(r) for r in train_recs]
print(train[0]["messages"][1]["content"][1]["text"])
print(train[0]["messages"][2]["content"][0]["text"][:500])

## 4. 4-bit base model + LoRA

In [ ]:
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(BASE, load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,     # box precision depends on the vision side, so it is trained too
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=SEED)

## 5. Held-out score (run now for the base model, again after training)
On held-out drafters: for each true symbol, the best IoU with a predicted box of the same kind (each prediction
used once); the share found at IoU >= 0.5; and how often the designator/value in the label is right.

In [ ]:
def predict(ex):
    img, text = ex["messages"][1]["content"][0]["image"], ex["messages"][1]["content"][1]["text"]
    msgs = [ex["messages"][0], {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(img, prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=60 * MAX_SYMBOLS, do_sample=False)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    boxes = []
    for m in re.finditer(r"\{[^{}]*\}", reply):
        try:
            o = json.loads(m.group(0)); boxes.append((str(o["label"]), [float(v) for v in o["bbox_2d"]]))
        except Exception:
            pass
    return boxes

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - ix * iy
    return ix * iy / u if u else 0

def evaluate(n=40):
    FastVisionModel.for_inference(model)
    scores, texts = [], []
    for rec in val_recs[:n]:
        pred, used = predict(example(rec)), set()
        for t in rec["boxes"]:
            cands = [(iou(t["bbox_2d"], p), i) for i, (l, p) in enumerate(pred)
                     if i not in used and l.lower().startswith(t["kind"])]
            s, i = max(cands, default=(0, None))
            scores.append(s)
            if i is not None and s >= 0.5:
                used.add(i)
                if t["label"] != t["kind"]:
                    texts.append(pred[i][0].strip() == t["label"])
    print(f"symbols {len(scores)}: mean IoU {sum(scores) / len(scores):.2f}, "
          f"found at IoU>=0.5 {sum(s >= 0.5 for s in scores) / len(scores):.0%}, "
          f"designator/value right {sum(texts) / max(1, len(texts)):.0%} of {len(texts)}")

evaluate()

## 6. Train
Training time per step on a T4 has not been measured for this setup: watch the first logged steps and multiply out.
If one epoch will not fit a session, set `PHOTOS_PER_DRAWING = 2` (re-run from section 2) or pass `max_steps`.
Checkpoints are saved every 200 steps; after a timeout, add `/kaggle/working/checkpoints` back and resume.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train,
    data_collator=UnslothVisionDataCollator(model, tokenizer, resize="max"),  # default "min" shrinks to 512 px
    args=SFTConfig(
        output_dir=f"{OUT}/checkpoints", per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS, learning_rate=1e-4, warmup_ratio=0.03, lr_scheduler_type="cosine",
        logging_steps=10, save_steps=200, save_total_limit=2, optim="adamw_8bit", fp16=True,
        seed=SEED, report_to="none", remove_unused_columns=False,
        dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True}, max_length=4096))   # TRL 0.22 name (formerly max_seq_length)
trainer.train()   # after a timeout: trainer.train(resume_from_checkpoint=True)

In [ ]:
evaluate()

## 7. Save

In [ ]:
model.save_pretrained(f"{OUT}/lora")            # the trained adapter (a few hundred MB): download this
tokenizer.save_pretrained(f"{OUT}/lora")
!cd /kaggle/working && zip -qr lora.zip lora

# The merged 16-bit model is about 17 GB and Kaggle keeps at most 20 GB of output, so it is off by default.
# To make it, set True: the prepared images and checkpoints are deleted first to make room.
SAVE_MERGED = False
if SAVE_MERGED:
    import shutil
    for d in ("prepared", "checkpoints"):
        shutil.rmtree(f"{OUT}/{d}", ignore_errors=True)
    model.save_pretrained_merged(f"{OUT}/merged", tokenizer, save_method="merged_16bit")

## 8. Load into Ollama (on your PC)
Download `lora.zip` (always saved) or `merged/` (only with `SAVE_MERGED = True`). The GGUF conversion below needs the merged model. With llama.cpp:

```
python convert_hf_to_gguf.py merged --outfile qwen3vl-circuits-f16.gguf
python convert_hf_to_gguf.py merged --mmproj --outfile mmproj-qwen3vl-circuits.gguf
llama-quantize qwen3vl-circuits-f16.gguf qwen3vl-circuits-Q4_K_M.gguf Q4_K_M
```

`Modelfile`:
```
FROM ./qwen3vl-circuits-Q4_K_M.gguf
FROM ./mmproj-qwen3vl-circuits.gguf
```

```
ollama create qwen3-vl-circuits -f Modelfile
python context/schematic-reader/scripts/analyze.py handwritten.jpeg --model qwen3-vl-circuits --max-side 1000
```

`--max-side` should equal `MAX_SIDE` above, so the model sees images at the scale it was trained on.
The fine-tune only learned the box question; the reader also uses the model to read the circuit, which the
LoRA may have shifted slightly, so compare a few known images before and after.
Converting Qwen3-VL fine-tunes (especially the vision projector) depends on your llama.cpp and Ollama versions.
If the model loads but ignores images, update both first.